# Everything Comes Back (Eventually): the Poincaré Recurrence Theorem

This notebook runs every simulation from the project and shows the figures.
Press **Runtime → Run all**. Nothing to install: Colab already has numpy, scipy and matplotlib.

Each section writes one script to disk with `%%writefile`, runs it, then displays what it made.
The full explanations are in `notes/physics_notes.md` in the repo.

In [ ]:
%%writefile style.py
"""Shared plot style so every figure in the deck looks like it came from one team."""
from pathlib import Path

import matplotlib as mpl
import matplotlib.pyplot as plt

FIG_DIR = Path("figures")
FIG_DIR.mkdir(exist_ok=True)

# A small, high-contrast palette that survives a washed-out projector.
INK = "#1b1f2a"
MUTED = "#6b7280"
BLUE = "#2563eb"
ORANGE = "#ea580c"
GREEN = "#059669"
PURPLE = "#7c3aed"
RED = "#dc2626"
PALETTE = [BLUE, ORANGE, GREEN, PURPLE, RED]


def apply():
    mpl.rcParams.update({
        "figure.dpi": 110,
        "savefig.dpi": 200,
        "savefig.bbox": "tight",
        "font.family": "DejaVu Sans",
        "font.size": 13,
        "axes.titlesize": 15,
        "axes.titleweight": "bold",
        "axes.labelsize": 13,
        "axes.edgecolor": INK,
        "axes.labelcolor": INK,
        "axes.spines.top": False,
        "axes.spines.right": False,
        "axes.prop_cycle": mpl.cycler(color=PALETTE),
        "xtick.color": INK,
        "ytick.color": INK,
        "text.color": INK,
        "legend.frameon": False,
        "mathtext.fontset": "cm",
    })


def save(fig, name):
    path = FIG_DIR / name
    fig.savefig(path)
    plt.close(fig)
    print(f"saved {path.relative_to(FIG_DIR.parent)}")
    return path

## 1. The proof in one picture
Copies of a disk under an area-keeping map. Finite room means they must overlap.

In [ ]:
%%writefile 01_proof_pigeonhole.py
"""
Figure for the proof slide: the pigeonhole idea.

We use the simplest area-preserving map we could think of: a shift on the
torus (a square box with wrap-around edges) by an irrational step

    f(q, p) = (q + a, p + b)  mod 1,   a = sqrt(5)-2, b = sqrt(2)-1

A disk U keeps its area under f. So every image f^n(U) has the same area.
The box has area 1. So at most about 1/area(U) disjoint copies fit.
Sooner or later two images must overlap, and (because f can be undone)
that means some f^k(U) overlaps U itself.

The same argument works for ANY volume-preserving map on a finite box,
including the flow of a Hamiltonian system. We just picked the one that
draws nicely.
"""
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Circle

import style

STEP = np.array([np.sqrt(5) - 2, np.sqrt(2) - 1])
CENTER = np.array([0.22, 0.25])
RADIUS = 0.06


def torus_dist(a, b):
    d = (a - b + 0.5) % 1.0 - 0.5
    return np.hypot(*d)


def first_overlap():
    """Smallest n >= 1 with f^n(U) touching U (two disks overlap iff centres < 2r apart)."""
    for n in range(1, 10_000):
        c = (CENTER + n * STEP) % 1.0
        if torus_dist(c, CENTER) < 2 * RADIUS:
            return n
    return None


def draw_disk(ax, c, **kw):
    # draw the disk plus its wrapped copies so blobs near an edge look right
    for dx in (-1, 0, 1):
        for dy in (-1, 0, 1):
            ax.add_patch(Circle(c + np.array([dx, dy]), RADIUS, **kw))


def main():
    style.apply()
    n_hit = first_overlap()
    area_ratio = np.pi * RADIUS**2
    print(f"area(U) = {area_ratio:.4f} of the box, so at most ~{int(1/area_ratio)} disjoint copies fit")
    print(f"first overlap with U at n = {n_hit}")

    fig, ax = plt.subplots(figsize=(7.2, 7.2))
    cmap = plt.get_cmap("viridis")
    for n in range(1, n_hit + 1):
        c = (CENTER + n * STEP) % 1.0
        last = n == n_hit
        col = style.ORANGE if last else cmap(0.15 + 0.7 * n / n_hit)
        draw_disk(ax, c, fc=col, ec="white", lw=1.0, alpha=0.85 if last else 0.6, zorder=3 if last else 2)
        ax.text(*c, str(n), ha="center", va="center", fontsize=12, color="white", weight="bold", zorder=6, clip_on=True)

    draw_disk(ax, CENTER, fc="none", ec=style.BLUE, lw=3, zorder=5)
    ax.text(CENTER[0] - RADIUS - 0.02, CENTER[1] + RADIUS + 0.01, "$U$", ha="right", fontsize=22,
            color=style.BLUE, weight="bold", zorder=6)

    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.set_aspect("equal")
    ax.set_xticks([])
    ax.set_yticks([])
    for s in ax.spines.values():
        s.set_visible(True)
        s.set_linewidth(2)
    ax.set_xlabel("position $q$")
    ax.set_ylabel("momentum $p$")
    ax.set_title("Every copy $f^n(U)$ has the same area as $U$")
    ax.text(0.5, -0.1,
            f"Copy {n_hit} lands on $U$. Finite room plus equal-sized guests means sharing.",
            transform=ax.transAxes, ha="center", fontsize=13, color=style.ORANGE)
    style.save(fig, "01_pigeonhole_proof.png")


if __name__ == "__main__":
    main()

In [ ]:
%run 01_proof_pigeonhole.py
from IPython.display import Image, display
display(Image(filename="figures/01_pigeonhole_proof.png"))

## 2. Liouville's theorem
A blob of pendulum states keeps its area. Add friction and it shrinks like $e^{-\gamma t}$.

In [ ]:
%%writefile 02_liouville_blob.py
"""
Why Hamiltonian systems obey the theorem: Liouville's theorem, seen directly.

We fill a small disk of initial conditions in the (q, p) plane of a pendulum

    H = p^2 / 2 - cos q,     dq/dt = p,   dp/dt = -sin q

and let every point move. The blob stretches into a thin swirl, but its
AREA never changes. Then we add friction (dp/dt = -sin q - gamma p).
Now the area shrinks like exp(-gamma t), every blob falls to the bottom,
and nothing ever comes back. No area conservation, no recurrence.

Outputs
  figures/02_liouville_blobs.png     snapshots, frictionless vs damped
  figures/02_area_vs_time.png        measured area against theory
  figures/02_liouville.gif           animation (for the live/HTML deck)
"""
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation, PillowWriter
from scipy.integrate import solve_ivp

import style

CENTER = np.array([0.0, 1.55])   # (q, p), sits just inside the separatrix p = 2 at q = 0
RADIUS = 0.45
GAMMA = 0.25                     # friction for the damped case
N_EDGE = 20000                   # points on the blob's boundary (for the area)
N_FILL = 6000                    # points inside (for the picture)
T_SNAP = [0.0, 3.0, 6.0, 12.0]
T_END = 12.0


def rhs(gamma):
    def f(t, y):
        n = y.size // 2
        q, p = y[:n], y[n:]
        return np.concatenate([p, -np.sin(q) - gamma * p])
    return f


def blob_points(rng):
    th = np.linspace(0, 2 * np.pi, N_EDGE, endpoint=False)
    edge = CENTER + RADIUS * np.column_stack([np.cos(th), np.sin(th)])
    ang = rng.uniform(0, 2 * np.pi, N_FILL)
    rad = RADIUS * np.sqrt(rng.uniform(0, 1, N_FILL))
    fill = CENTER + np.column_stack([rad * np.cos(ang), rad * np.sin(ang)])
    return edge, fill


def evolve(points, gamma, t_eval):
    y0 = np.concatenate([points[:, 0], points[:, 1]])
    sol = solve_ivp(rhs(gamma), (0, t_eval[-1]), y0, t_eval=t_eval,
                    method="DOP853", rtol=1e-10, atol=1e-12)
    n = points.shape[0]
    return sol.y[:n].T, sol.y[n:].T          # shape (len(t), n)


def shoelace(q, p):
    return 0.5 * abs(np.dot(q, np.roll(p, -1)) - np.dot(p, np.roll(q, -1)))


def wrap(q):
    return (q + np.pi) % (2 * np.pi) - np.pi


def phase_portrait(ax):
    qg, pg = np.meshgrid(np.linspace(-np.pi, np.pi, 400), np.linspace(-3, 3, 400))
    H = 0.5 * pg**2 - np.cos(qg)
    ax.contour(qg, pg, H, levels=np.linspace(-0.9, 3, 12), colors="#d1d5db", linewidths=0.7,
               negative_linestyles="solid")
    ax.contour(qg, pg, H, levels=[1.0], colors=style.MUTED, linewidths=1.2, linestyles="--")


def main():
    style.apply()
    rng = np.random.default_rng(0)
    edge, fill = blob_points(rng)
    A0 = np.pi * RADIUS**2

    t_area = np.linspace(0, T_END, 61)
    results = {}
    for gamma in (0.0, GAMMA):
        eq, ep = evolve(edge, gamma, t_area)
        fq, fp = evolve(fill, gamma, np.array(T_SNAP) if T_SNAP[0] > 0 else np.array(T_SNAP))
        areas = np.array([shoelace(eq[i], ep[i]) for i in range(len(t_area))])
        results[gamma] = dict(fq=fq, fp=fp, areas=areas)
        print(f"gamma={gamma}: area at t=0 {areas[0]:.5f}, at t={T_END} {areas[-1]:.5f}, "
              f"theory {A0*np.exp(-gamma*T_END):.5f}")

    # ---- snapshots ----
    fig, axes = plt.subplots(2, len(T_SNAP), figsize=(15, 7.4), sharex=True, sharey=True)
    for row, gamma in enumerate((0.0, GAMMA)):
        r = results[gamma]
        colour = style.BLUE if gamma == 0 else style.RED
        for col, t in enumerate(T_SNAP):
            ax = axes[row, col]
            phase_portrait(ax)
            i = int(np.argmin(np.abs(np.array(T_SNAP) - t)))
            ax.scatter(wrap(r["fq"][i]), r["fp"][i], s=0.8, color=colour, linewidths=0)
            ia = int(np.argmin(np.abs(t_area - t)))
            ax.set_title(f"$t = {t:g}$    area $= {r['areas'][ia]:.3f}$", fontsize=12.5)
            ax.set_xlim(-np.pi, np.pi)
            ax.set_ylim(-3, 3)
            ax.set_xticks([-np.pi, 0, np.pi], ["$-\\pi$", "0", "$\\pi$"])
            if col == 0:
                ax.set_ylabel("$p$")
            if row == 1:
                ax.set_xlabel("$q$")
    fig.text(0.005, 0.73, "No friction\n(Hamiltonian)", rotation=90, va="center", ha="left",
             fontsize=14, weight="bold", color=style.BLUE)
    fig.text(0.005, 0.29, f"With friction\n($\\gamma = {GAMMA}$)", rotation=90, va="center", ha="left",
             fontsize=14, weight="bold", color=style.RED)
    fig.suptitle("A blob of pendulums: it may stretch, but without friction its area stays fixed",
                 fontsize=16, weight="bold")
    fig.tight_layout(rect=(0.03, 0, 1, 0.96))
    style.save(fig, "02_liouville_blobs.png")

    # ---- compact 2 x 3 version for the slide ----
    t_slide = [0.0, 6.0, 12.0]
    fig, axes = plt.subplots(2, 3, figsize=(9.6, 6.6), sharex=True, sharey=True)
    for row, gamma in enumerate((0.0, GAMMA)):
        r = results[gamma]
        colour = style.BLUE if gamma == 0 else style.RED
        for col, t in enumerate(t_slide):
            ax = axes[row, col]
            phase_portrait(ax)
            i = T_SNAP.index(t)
            ax.scatter(wrap(r["fq"][i]), r["fp"][i], s=1.0, color=colour, linewidths=0)
            ia = int(np.argmin(np.abs(t_area - t)))
            ax.set_title(f"$t={t:g}$, area $={r['areas'][ia]:.3f}$", fontsize=13, color=colour)
            ax.set_xlim(-np.pi, np.pi)
            ax.set_ylim(-3, 3)
            ax.set_xticks([])
            ax.set_yticks([])
            for sp in ax.spines.values():
                sp.set_visible(True)
                sp.set_color("#9ca3af")
    axes[0, 0].set_ylabel("no friction", fontsize=15, color=style.BLUE, weight="bold")
    axes[1, 0].set_ylabel("with friction", fontsize=15, color=style.RED, weight="bold")
    fig.tight_layout(h_pad=1.2, w_pad=0.6)
    style.save(fig, "02_liouville_slide.png")

    # ---- area vs time ----
    fig, ax = plt.subplots(figsize=(7, 4.2))
    ax.plot(t_area, results[0.0]["areas"] / A0, "o", ms=4, color=style.BLUE, label="no friction (measured)")
    ax.axhline(1.0, color=style.BLUE, lw=1, alpha=0.5)
    ax.plot(t_area, results[GAMMA]["areas"] / A0, "o", ms=4, color=style.RED, label="friction (measured)")
    ax.plot(t_area, np.exp(-GAMMA * t_area), color=style.RED, lw=1.5, alpha=0.7,
            label=f"theory $e^{{-\\gamma t}}$")
    ax.set_xlabel("time $t$")
    ax.set_ylabel("area / starting area")
    ax.set_ylim(0, 1.15)
    ax.legend(loc="center right")
    ax.set_title("Liouville's theorem, checked numerically")
    style.save(fig, "02_area_vs_time.png")

    # ---- gif (frictionless only) ----
    t_gif = np.linspace(0, T_END, 90)
    gq, gp = evolve(fill, 0.0, t_gif)
    fig, ax = plt.subplots(figsize=(5.5, 5.2))
    phase_portrait(ax)
    sc = ax.scatter(wrap(gq[0]), gp[0], s=1.2, color=style.BLUE, linewidths=0)
    ttl = ax.set_title("")
    ax.set_xlim(-np.pi, np.pi)
    ax.set_ylim(-3, 3)
    ax.set_xlabel("$q$")
    ax.set_ylabel("$p$")

    def update(k):
        sc.set_offsets(np.column_stack([wrap(gq[k]), gp[k]]))
        ttl.set_text(f"pendulum phase space, t = {t_gif[k]:.1f}")
        return sc, ttl

    anim = FuncAnimation(fig, update, frames=len(t_gif), blit=False)
    anim.save(style.FIG_DIR / "02_liouville.gif", writer=PillowWriter(fps=15), dpi=80)
    plt.close(fig)
    print("saved figures/02_liouville.gif")


if __name__ == "__main__":
    main()

In [ ]:
%run 02_liouville_blob.py
from IPython.display import Image, display
display(Image(filename="figures/02_liouville_blobs.png"))
display(Image(filename="figures/02_area_vs_time.png"))
display(Image(filename="figures/02_liouville.gif"))

## 3. Example 1: two oscillators with frequency ratio $\varphi$
Takes about a minute (the return-time scan is the slow part).

In [ ]:
%%writefile 03_torus_oscillators.py
"""
Example 1: two uncoupled oscillators with an irrational frequency ratio.

    H = (p1^2 + w1^2 x1^2)/2 + (p2^2 + w2^2 x2^2)/2,   w2 / w1 = golden ratio

Each oscillator keeps its own energy, so the state lives on a 2-torus
labelled by two phase angles (theta1, theta2) = (w1 t, w2 t). Because the
ratio is irrational the motion never exactly repeats. Yet it keeps coming
back as close as we like to where it started. That is Poincare recurrence
in its friendliest form.

Outputs
  figures/03_torus.png              trajectory winding on a doughnut
  figures/03_distance.png           distance from the start vs time, with returns
  figures/03_return_scaling.png     how the return time grows as we ask for a closer return
"""
import numpy as np
import matplotlib.pyplot as plt

import style

PHI = (1 + 5**0.5) / 2
W = np.array([1.0, PHI])


def torus_distance(theta, theta0):
    d = (theta - theta0 + np.pi) % (2 * np.pi) - np.pi
    return np.sqrt(np.sum(d**2, axis=-1))


def mean_return_time(omegas, eps, t_total, chunk=2_000_000):
    """Average time between visits to the eps-ball around the start.

    We run for a long time and count how many times the phase point ENTERS
    the ball. Mean return time = total time / number of entries.
    """
    dt = eps / 6
    entries, inside_prev, t0 = 0, True, 0.0
    while t0 < t_total:
        t = t0 + dt * np.arange(chunk)
        inside = torus_distance(np.outer(t, omegas), 0.0) < eps
        prev = np.concatenate([[inside_prev], inside[:-1]])
        entries += np.count_nonzero(inside & ~prev)
        inside_prev = inside[-1]
        t0 = t[-1] + dt
    return t0 / entries if entries else np.nan


def theory_return_time(omegas, eps):
    """Flux argument: torus volume / (speed x cross-section of the eps-ball)."""
    n = len(omegas)
    speed = np.linalg.norm(omegas)
    cross = 2 * eps if n == 2 else np.pi * eps**2      # 1D 'diameter' or 2D disk
    return (2 * np.pi) ** n / (speed * cross)


def draw_torus(ax, t_end=60):
    R, r = 3.0, 1.1
    elev, azim = 38, -60
    u, v = np.meshgrid(np.linspace(0, 2 * np.pi, 80), np.linspace(0, 2 * np.pi, 40))
    X = (R + r * np.cos(v)) * np.cos(u)
    Y = (R + r * np.cos(v)) * np.sin(u)
    Z = r * np.sin(v)
    ax.plot_surface(X, Y, Z, color="#dbe3f0", alpha=0.55, linewidth=0, shade=True, zorder=1)

    t = np.linspace(0, t_end, 8000)
    th1, th2 = W[0] * t, W[1] * t
    x = (R + r * np.cos(th2)) * np.cos(th1)
    y = (R + r * np.cos(th2)) * np.sin(th1)
    z = r * np.sin(th2)
    # crude depth cue: is the surface normal pointing at the camera?
    e, a = np.radians(elev), np.radians(azim)
    view = np.array([np.cos(e) * np.cos(a), np.cos(e) * np.sin(a), np.sin(e)])
    normal = np.stack([np.cos(th2) * np.cos(th1), np.cos(th2) * np.sin(th1), np.sin(th2)])
    front = view @ normal > 0
    xb, yb, zb = (np.where(front, np.nan, c) for c in (x, y, z))
    xf, yf, zf = (np.where(front, c, np.nan) for c in (x, y, z))
    ax.plot(xb, yb, zb, color=style.BLUE, lw=0.8, alpha=0.25, zorder=2)
    ax.plot(xf, yf, zf, color=style.BLUE, lw=1.3, alpha=0.95, zorder=3)
    ax.scatter([R + r], [0], [0], s=90, color=style.ORANGE, zorder=10, depthshade=False)
    ax.set_box_aspect((1, 1, 0.38), zoom=1.35)
    ax.set_axis_off()
    ax.view_init(elev=elev, azim=azim)
    ax.computed_zorder = False


def draw_unfolded(ax, t_end=60):
    t = np.linspace(0, t_end, 20000)
    th = (np.outer(t, W)) % (2 * np.pi)
    jump = np.any(np.abs(np.diff(th, axis=0)) > np.pi, axis=1)
    th = th.copy()
    th[1:][jump] = np.nan
    ax.plot(th[:, 0], th[:, 1], color=style.BLUE, lw=1.0)
    ax.plot([0], [0], "o", ms=11, color=style.ORANGE, clip_on=False, zorder=5)
    ax.set_xlim(0, 2 * np.pi)
    ax.set_ylim(0, 2 * np.pi)
    ax.set_aspect("equal")
    ticks = [0, np.pi, 2 * np.pi]
    labels = ["0", "$\\pi$", "$2\\pi$"]
    ax.set_xticks(ticks, labels)
    ax.set_yticks(ticks, labels)
    ax.set_xlabel("phase of oscillator 1, $\\theta_1$")
    ax.set_ylabel("phase of oscillator 2, $\\theta_2$")
    for sp in ax.spines.values():
        sp.set_visible(True)


def main():
    style.apply()

    # ---- torus picture: doughnut + the same thing cut open ----
    fig = plt.figure(figsize=(12, 5.0))
    ax3 = fig.add_subplot(1, 2, 1, projection="3d")
    draw_torus(ax3)
    ax3.set_title("on the doughnut")
    ax2 = fig.add_subplot(1, 2, 2)
    draw_unfolded(ax2)
    ax2.set_title("cut open into a square")
    style.save(fig, "03_torus.png")

    # ---- distance from start ----
    t = np.linspace(0, 400, 400_000)
    d = torus_distance(np.outer(t, W), 0.0)
    eps = 0.15
    fig, ax = plt.subplots(figsize=(10, 3.8))
    ax.plot(t, d, lw=0.6, color=style.BLUE)
    ax.axhline(eps, color=style.ORANGE, lw=1.5, ls="--")
    ax.text(402, eps, f"  $\\varepsilon = {eps}$", color=style.ORANGE, va="center")
    close = d < eps
    starts = np.nonzero(close & ~np.roll(close, 1))[0]
    ax.plot(t[starts[1:]], d[starts[1:]], "v", color=style.ORANGE, ms=9)
    ax.set_xlim(0, 400)
    ax.set_xlabel("time $t$  (units of $1/\\omega_1$)")
    ax.set_ylabel("distance from start")
    ax.set_title("It keeps coming back: every orange marker is a return within $\\varepsilon$")
    style.save(fig, "03_distance.png")
    print(f"returns within eps={eps} up to t=400: {len(starts) - 1}")

    # ---- return-time scaling with eps (2 and 3 oscillators) ----
    fig, ax = plt.subplots(figsize=(6.6, 4.6))
    configs = [
        (np.array([1.0, PHI]), np.geomspace(0.4, 0.01, 7), style.BLUE, "2 oscillators", 1, 1.5e5),
        (np.array([1.0, PHI, 2**0.5]), np.geomspace(0.5, 0.08, 6), style.GREEN, "3 oscillators", 2, 1.5e6),
    ]
    for om, epss, col, lab, slope, t_total in configs:
        T = np.array([mean_return_time(om, e, t_total) for e in epss])
        Th = theory_return_time(om, epss)
        print(lab, "measured", np.round(T, 1), "theory", np.round(Th, 1))
        ax.loglog(epss, T, "o", color=col, label=f"{lab} (simulated)", ms=7)
        ax.loglog(epss, Th, "-", color=col, alpha=0.6, label=f"theory $\\propto \\varepsilon^{{-{slope}}}$")
    ax.invert_xaxis()
    ax.set_xlabel("how close we demand: $\\varepsilon$")
    ax.set_ylabel("mean return time $T$")
    ax.set_title("Each extra oscillator costs a factor $1/\\varepsilon$")
    ax.legend(fontsize=11)
    style.save(fig, "03_return_scaling.png")


if __name__ == "__main__":
    main()

In [ ]:
%run 03_torus_oscillators.py
from IPython.display import Image, display
display(Image(filename="figures/03_torus.png"))
display(Image(filename="figures/03_distance.png"))
display(Image(filename="figures/03_return_scaling.png"))

## 4. Example 2: Arnold's cat map
Chaotic, yet the picture returns exactly.

In [ ]:
%%writefile 04_cat_map.py
"""
Example 2: Arnold's cat map, the most dramatic recurrence there is.

On an N x N grid of pixels, move the pixel at (x, y) to

    (x, y)  ->  (2x + y,  x + y)   mod N

The matrix [[2, 1], [1, 1]] has determinant 1, so the map is a one-to-one
shuffle of pixels: it preserves "area" (pixel count) exactly. It is also
chaotic: nearby pixels fly apart fast. The picture turns to static.
But a shuffle of a FINITE set of pixels must repeat. For N = 101 the
picture comes back perfectly after exactly 25 steps.

Outputs
  figures/04_cat_map_strip.png   selected frames in a row
  figures/04_cat_map.gif         the full loop
"""
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation, PillowWriter

import style

N = 101


def make_picture(n=N):
    """A home-made test picture: a smiling face with a pendulum next to it."""
    y, x = np.mgrid[0:n, 0:n] / n
    img = np.ones((n, n, 3)) * np.array([0.93, 0.95, 0.99])          # pale background
    face = (x - 0.38) ** 2 + (y - 0.45) ** 2 < 0.27**2
    img[face] = [0.99, 0.80, 0.20]
    for ex in (0.29, 0.47):                                           # eyes
        img[(x - ex) ** 2 + (y - 0.37) ** 2 < 0.045**2] = [0.11, 0.12, 0.16]
    r = np.hypot(x - 0.38, y - 0.47)                                  # smile
    img[(np.abs(r - 0.15) < 0.025) & (y > 0.52) & face] = [0.11, 0.12, 0.16]
    # pendulum on the right: a string and a blue bob
    img[(np.abs(x - 0.82) < 0.012) & (y > 0.08) & (y < 0.66)] = [0.42, 0.45, 0.50]
    img[(x - 0.82) ** 2 + (y - 0.74) ** 2 < 0.09**2] = [0.15, 0.39, 0.92]
    img[(y < 0.08) & (x > 0.68) & (x < 0.96)] = [0.11, 0.12, 0.16]    # ceiling
    return img


def cat(img):
    n = img.shape[0]
    ys, xs = np.mgrid[0:n, 0:n]
    out = np.empty_like(img)
    out[(xs + ys) % n, (2 * xs + ys) % n] = img[ys, xs]               # row index = y
    return out


def main():
    style.apply()
    frames = [make_picture()]
    for _ in range(60):
        frames.append(cat(frames[-1]))
        if np.array_equal(frames[-1], frames[0]):
            break
    period = len(frames) - 1
    print(f"cat map on a {N}x{N} grid returns exactly after {period} steps")

    show = [0, 1, 2, 4, 12, 20, 24, period]
    fig, axes = plt.subplots(1, len(show), figsize=(2.1 * len(show), 2.75))
    for ax, k in zip(axes, show):
        ax.imshow(frames[k], interpolation="nearest")
        ax.set_xticks([])
        ax.set_yticks([])
        last = k == period
        col = style.ORANGE if last else style.INK
        ax.set_title(f"step {k}", color=col, fontsize=13)
        for sp in ax.spines.values():
            sp.set_visible(True)
            sp.set_color(style.ORANGE if last else style.MUTED)
            sp.set_linewidth(3 if last else 0.8)
    fig.suptitle(f"Arnold's cat map on {N}$\\times${N} pixels: scrambled to noise, "
                 f"then back exactly at step {period}", fontsize=14, weight="bold", y=1.04)
    style.save(fig, "04_cat_map_strip.png")

    fig, ax = plt.subplots(figsize=(4, 4.3))
    im = ax.imshow(frames[0], interpolation="nearest")
    ax.set_xticks([])
    ax.set_yticks([])
    ttl = ax.set_title("step 0")

    def update(k):
        im.set_data(frames[k])
        ttl.set_text(f"step {k}" + ("  (back!)" if k == period else ""))
        return im, ttl

    order = list(range(period + 1)) + [period] * 8          # pause on the return
    anim = FuncAnimation(fig, lambda i: update(order[i]), frames=len(order))
    anim.save(style.FIG_DIR / "04_cat_map.gif", writer=PillowWriter(fps=4), dpi=90)
    plt.close(fig)
    print("saved figures/04_cat_map.gif")


if __name__ == "__main__":
    main()

In [ ]:
%run 04_cat_map.py
from IPython.display import Image, display
display(Image(filename="figures/04_cat_map_strip.png"))
display(Image(filename="figures/04_cat_map.gif"))

## 5. Main numerical case: gas in a box
All particles start on the left. Kac's lemma predicts a mean wait of $2^N$ looks. About a minute.

In [ ]:
%%writefile 05_gas_in_box.py
"""
MAIN NUMERICAL CASE: a gas in a box, and how long it takes to "un-mix".

Setup
  N non-interacting particles in a square box of side 1, bouncing off the
  walls (elastic). All start in the LEFT half. We watch them spread out
  and ask: when are they all back in the left half?

Why this is a Poincare system
  Each particle's x-motion is a bounce between walls. Unfold the bounce and
  it becomes a point running round a circle of length 2:
      theta(t) = theta0 + v t  (mod 2),     x = theta if theta < 1 else 2 - theta
  So the full x-state is a point on an N-dimensional torus, moving in a
  straight line. That flow preserves volume and the torus is finite, so
  Poincare says the gas MUST come back to the left half, again and again.

What we measure (the prediction we test)
  Look at the box once per time step. Let A = "all N particles in the left
  half". A takes up a fraction mu(A) = (1/2)^N of the torus. Kac's lemma
  (the quantitative version of Poincare) says the AVERAGE number of steps
  between visits to A is exactly

      <T> = 1 / mu(A) = 2^N.

  We simulate long runs, measure <T> for N = 1..14, and compare.

Outputs
  figures/05_gas_snapshots.png   the box at a few moments, ending at a return
  figures/05_left_fraction.png   fraction of particles on the left vs time, N=10 vs N=40
  figures/05_kac_scaling.png     measured mean return time vs 2^N
  figures/05_gas.gif             animation of the spread and the return
  figures/05_results.csv         the numbers behind the Kac plot
"""
import csv

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation, PillowWriter
from matplotlib.patches import Rectangle

import style

DT = 1.0                 # we look at the box once per unit of time
RNG_SEED = 7


# ---------------------------------------------------------------- physics
def bounce(theta):
    """Unfolded coordinate on a circle of length 2 -> position in the box [0, 1]."""
    th = theta % 2.0
    return np.where(th < 1.0, th, 2.0 - th)


def make_gas(n, rng):
    """Random start with everyone in the left half, Maxwell-like random velocities."""
    x0 = rng.uniform(0, 0.5, n)
    y0 = rng.uniform(0, 1.0, n)
    vx = rng.normal(0, 1, n)
    vy = rng.normal(0, 1, n)
    return x0, y0, vx, vy


def positions(x0, y0, vx, vy, t):
    t = np.atleast_1d(t)[:, None]
    return bounce(x0 + vx * t), bounce(y0 + vy * t)


def visits_to_left_half(x0, vx, n_steps, chunk=1_000_000):
    """Step indices k (k >= 1) at which ALL particles have x < 1/2."""
    hits = []
    for start in range(1, n_steps + 1, chunk):
        k = np.arange(start, min(start + chunk, n_steps + 1))
        x = bounce(x0[None, :] + vx[None, :] * (k[:, None] * DT))
        hits.append(k[np.all(x < 0.5, axis=1)])
    return np.concatenate(hits)


# ---------------------------------------------------------------- figures
def draw_box(ax, x, y, t, highlight=False):
    ax.add_patch(Rectangle((0, 0), 0.5, 1, fc=style.BLUE, alpha=0.08, lw=0))
    ax.axvline(0.5, color=style.MUTED, lw=1, ls=":")
    left = x < 0.5
    ax.scatter(x[left], y[left], s=60, color=style.BLUE, edgecolor="white", lw=0.8, zorder=3)
    ax.scatter(x[~left], y[~left], s=60, color=style.ORANGE, edgecolor="white", lw=0.8, zorder=3)
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.set_aspect("equal")
    ax.set_xticks([])
    ax.set_yticks([])
    for sp in ax.spines.values():
        sp.set_visible(True)
        sp.set_linewidth(3 if highlight else 1.5)
        sp.set_color(style.GREEN if highlight else style.INK)
    ax.set_title(f"t = {t:g}" + ("   all back!" if highlight else ""),
                 color=style.GREEN if highlight else style.INK, fontsize=13)


def fig_snapshots(rng):
    n = 10
    x0, y0, vx, vy = make_gas(n, rng)
    k_ret = int(visits_to_left_half(x0, vx, 200_000)[0])
    print(f"[snapshots] N={n}: first return to the left half at step {k_ret} (Kac average 2^{n} = {2**n})")
    times = [0, 0.5, 2, 30, k_ret]
    fig, axes = plt.subplots(1, len(times), figsize=(3.0 * len(times), 3.3))
    for ax, t in zip(axes, times):
        x, y = positions(x0, y0, vx, vy, t)
        draw_box(ax, x[0], y[0], t, highlight=(t == k_ret))
    fig.suptitle(f"{n} particles start in the left half. They spread out. "
                 f"After {k_ret} looks, they are all back on the left.",
                 fontsize=14, weight="bold", y=1.03)
    style.save(fig, "05_gas_snapshots.png")
    return (x0, y0, vx, vy), k_ret


def fig_left_fraction(rng):
    t = np.arange(0, 6001) * DT
    fig, axes = plt.subplots(2, 1, figsize=(11, 5.2), sharex=True)
    for ax, n, col in zip(axes, (10, 40), (style.BLUE, style.PURPLE)):
        x0, _, vx, _ = make_gas(n, rng)
        x = bounce(x0[None, :] + vx[None, :] * t[:, None])
        frac = np.mean(x < 0.5, axis=1)
        ax.plot(t, frac, lw=0.5, color=col)
        back = np.nonzero(frac[1:] == 1.0)[0] + 1
        ax.plot(t[back], frac[back], "v", color=style.GREEN, ms=8)
        ax.axhline(0.5, color=style.MUTED, lw=1, ls=":")
        ax.set_ylim(0, 1.12)
        ax.set_ylabel("fraction on left")
        msg = (f"N = {n}: {len(back)} returns (green) in {len(t)-1} looks"
               if len(back) else f"N = {n}: no return in {len(t)-1} looks "
                                 f"(expected wait about $2^{{{n}}} \\approx 10^{{{int(round(n*np.log10(2)))}}}$)")
        ax.set_title(msg, loc="left", fontsize=13)
    axes[-1].set_xlabel("time (number of looks)")
    fig.tight_layout()
    style.save(fig, "05_left_fraction.png")


def fig_kac(rng):
    Ns = np.arange(1, 15)
    rows = []
    for n in Ns:
        x0, _, vx, _ = make_gas(n, rng)
        n_steps = int(min(4000 * 2.0**n, 6e7))
        hits = visits_to_left_half(x0, vx, n_steps)
        gaps = np.diff(hits)
        mean = gaps.mean()
        err = gaps.std(ddof=1) / np.sqrt(len(gaps))
        rows.append((int(n), len(gaps), mean, err, 2.0**n))
        print(f"N={n:2d}  returns={len(gaps):7d}  mean gap={mean:10.1f} +/- {err:7.1f}   Kac 2^N={2**n}")

    with open(style.FIG_DIR / "05_results.csv", "w", newline="") as f:
        w = csv.writer(f)
        w.writerow(["N", "returns_counted", "mean_return_steps", "std_error", "kac_prediction"])
        w.writerows(rows)

    r = np.array(rows)
    fig, ax = plt.subplots(figsize=(7.6, 5))
    nn = np.linspace(1, 14, 100)
    ax.semilogy(nn, 2.0**nn, color=style.ORANGE, lw=2.5, label="Kac's prediction $2^N$", zorder=1)
    ax.errorbar(r[:, 0], r[:, 2], yerr=r[:, 3], fmt="o", ms=8, color=style.BLUE,
                capsize=3, label="our simulation", zorder=2)
    ax.set_xlabel("number of particles $N$")
    ax.set_ylabel("mean wait between returns (looks)")
    ax.set_title("Waiting time doubles with every particle")
    ax.legend(loc="upper left")
    ax.set_xticks(Ns)
    style.save(fig, "05_kac_scaling.png")

    # The punchline numbers
    ms_per_look = 1e-3
    year = 3.156e7
    for n in (100, 1000):
        yrs = 2.0**n * ms_per_look / year
        print(f"N={n}: 2^N looks at one per millisecond = {yrs:.2e} years "
              f"= {yrs/1.38e10:.1e} x age of universe")
    return rows


def gif_gas(gas, k_ret):
    x0, y0, vx, vy = gas
    early = np.linspace(0, 6, 60)
    late = np.linspace(k_ret - 6, k_ret, 60)
    times = list(early) + [None] * 6 + list(late) + [k_ret] * 12
    fig, ax = plt.subplots(figsize=(4.2, 4.5))

    def update(i):
        ax.clear()
        t = times[i]
        if t is None:
            ax.text(0.5, 0.5, f"... fast forward\nto t = {k_ret - 6:g} ...", ha="center",
                    va="center", fontsize=15, transform=ax.transAxes)
            ax.set_xticks([]), ax.set_yticks([])
            return
        x, y = positions(x0, y0, vx, vy, t)
        draw_box(ax, x[0], y[0], round(t, 1), highlight=bool(np.all(x[0] < 0.5) and t > 1))

    anim = FuncAnimation(fig, update, frames=len(times))
    anim.save(style.FIG_DIR / "05_gas.gif", writer=PillowWriter(fps=12), dpi=80)
    plt.close(fig)
    print("saved figures/05_gas.gif")


def main():
    style.apply()
    rng = np.random.default_rng(RNG_SEED)
    gas, k_ret = fig_snapshots(rng)
    fig_left_fraction(rng)
    fig_kac(rng)
    gif_gas(gas, k_ret)


if __name__ == "__main__":
    main()

In [ ]:
%run 05_gas_in_box.py
from IPython.display import Image, display
display(Image(filename="figures/05_gas_snapshots.png"))
display(Image(filename="figures/05_left_fraction.png"))
display(Image(filename="figures/05_kac_scaling.png"))
display(Image(filename="figures/05_gas.gif"))

## Try it yourself
* In `05_gas_in_box.py`, change `n = 10` in `fig_snapshots` to 12 or 15 and see how long the gas takes to come back.
* In `04_cat_map.py`, change `N = 101` to 99 or 150. The period jumps around a lot.
* In `02_liouville_blob.py`, raise `GAMMA` and watch the blob die faster.